## Install Dependencies

In [1]:
import os
os.environ['CUDA_HOME'] = '/usr/local/cuda'
os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'

!pip install -q torch==2.3.0 torchvision==0.18.0 --index-url https://download.pytorch.org/whl/cu121

import torch
print(f"Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

Torch: 2.3.0+cu121, CUDA: True


In [2]:
!apt-get install -q -y ninja-build
!git clone https://github.com/IDEA-Research/GroundingDINO.git
%cd GroundingDINO
!pip install -q -e .
%cd ..

!pip install -q git+https://github.com/facebookresearch/segment-anything.git
!pip install -q opencv-python-headless Pillow numpy matplotlib playwright
!playwright install chromium

!mkdir -p weights
!wget -q -O weights/groundingdino_swint_ogc.pth \
    https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth
!wget -q -O weights/sam_vit_h_4b8939.pth \
    https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth


Reading package lists...
Building dependency tree...
Reading state information...
ninja-build is already the newest version (1.10.1-1).
0 upgraded, 0 newly installed, 0 to remove and 100 not upgraded.
fatal: destination path 'GroundingDINO' already exists and is not an empty directory.
/content/GroundingDINO
  Preparing metadata (setup.py) ... done
/content
  Preparing metadata (setup.py) ... done


In [15]:
!pip install -q fiftyone playwright
!playwright install-deps
!playwright install chromium

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.7/17.7 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.5/323.5 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.9/112.9 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 116.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 11.5 MB/s eta

#Configuration

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import fiftyone as fo
import fiftyone.zoo as foz
import shutil, os, json
from pathlib import Path

BASE = '/content/drive/MyDrive/augle_ai'

CFG = {
    'output_dir': f'{BASE}/data/images',
    'coco_path':  f'{BASE}/data/annotations/instances.json',
    'image_size':      (640, 640),
    'gd_config':       'GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py',
    'gd_weights':      'weights/groundingdino_swint_ogc.pth',
    'sam_checkpoint':  'weights/sam_vit_h_4b8939.pth',
    'sam_model_type':  'vit_h',
    'gd_text_prompt':  'person . car . dog .',
    'gd_box_threshold': 0.35,
    'gd_text_threshold': 0.25,

    # 3 categories × 100 Playwright + 200 FiftyOne = 900 total
    'categories': ['person', 'car', 'dog'],
    'pw_cat': 100,   # 300 total from Unsplash
    'fone_cat':   200,   # 600 total from Open Images V7

    # Multiple search queries per category so Unsplash has variety
    'search_queries': {
        'person': ['street photography people', 'crowd walking city',
                   'person portrait outdoor', 'people market',
                   'human activity park'],
        'car':    ['cars on road traffic', 'car parking lot',
                   'automobile highway', 'vehicles city street',
                   'car close up'],
        'dog':    ['dog outdoor park', 'dog running field',
                   'puppy playing', 'dog portrait', 'dog fetch ball'],
    },

    'fiftyone_classes': {
        'person': 'Person',
        'car':    'Car',
        'dog':    'Dog',
    },
}

print('Config ready.')
print(f"Target: {len(CFG['categories'])} categories × "
      f"({CFG['pw_cat']} Playwright + {CFG['fo_cat']} FiftyOne) "
      f"= {len(CFG['categories']) * (CFG['pw_cat'] + CFG['fo_cat'])} images")

Config ready.
Target: 3 categories × (100 Playwright + 200 FiftyOne) = 900 images


### Image Collection — Playwright (Unsplash)

In [4]:
import asyncio, urllib.request, time
from pathlib import Path
from playwright.async_api import async_playwright

async def scrape_unsplash(query: str, n: int, save_dir: Path) -> list[str]:
    save_dir.mkdir(parents=True, exist_ok=True)
    saved_paths: list[str] = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=['--no-sandbox', '--disable-setuid-sandbox']
        )
        context = await browser.new_context(
            user_agent='Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                       'AppleWebKit/537.36 (KHTML, like Gecko) '
                       'Chrome/119.0.0.0 Safari/537.36'
        )
        page = await context.new_page()
        search_url = f"https://unsplash.com/s/photos/{query.replace(' ', '+')}"
        await page.goto(search_url, timeout=60_000, wait_until='networkidle')
        await page.wait_for_timeout(3000)

        hrefs = set()
        for _ in range(20):
            imgs = await page.query_selector_all('img[alt]')
            for img in imgs:
                src = await img.get_attribute('src')
                if src and 'images.unsplash.com' in src and 'profile' not in src:
                    hrefs.add(src.split('?')[0] + '?w=640&q=80&fm=jpg')
            if len(hrefs) >= n:
                break
            await page.evaluate('window.scrollBy(0, window.innerHeight * 2)')
            await page.wait_for_timeout(1500)

        await browser.close()

    headers = {'User-Agent': 'Mozilla/5.0'}
    for idx, href in enumerate(list(hrefs)[:n]):
        fname = save_dir / f"{query.replace(' ', '_')}_{idx:04d}.jpg"
        if fname.exists():
            saved_paths.append(str(fname)); continue
        try:
            req = urllib.request.Request(href, headers=headers)
            with urllib.request.urlopen(req, timeout=15) as resp:
                data = resp.read()
            with open(fname, 'wb') as f:
                f.write(data)
            saved_paths.append(str(fname))
            time.sleep(0.2)
        except Exception as e:
            print(f' {href}: {e}')

    print(f'  [Playwright] "{query}" → {len(saved_paths)} images')
    return saved_paths


async def collect_pw(cfg: dict) -> list[str]:
    # Collect playwright images for each category.
    all_paths = []
    per_cat   = cfg['pw_cat']

    for category in cfg['categories']:
        queries  = cfg['search_queries'][category]
        cat_dir  = Path(cfg['output_dir']) / 'unsplash' / category
        per_q    = max(1, per_cat // len(queries)) + 5   # +5 buffer for failed downloads
        cat_paths = []

        print(f'\n[Playwright] Category: {category}  target={per_cat}')
        for query in queries:
            if len(cat_paths) >= per_cat:
                break
            paths = await scrape_unsplash(query, per_q, cat_dir)
            cat_paths.extend(paths)

        cat_paths = cat_paths[:per_cat]
        print(f'  → {len(cat_paths)} images collected for [{category}]')
        all_paths.extend(cat_paths)

    print(f'\n[Playwright] Total: {len(all_paths)} images')
    return all_paths


print('Running Playwright collection...')
pw_paths = await collect_pw(CFG)
all_image_paths  = pw_paths.copy()
print(f'Playwright done: {len(pw_paths)} images')

Running Playwright collection...

[Playwright] Category: person  target=100
  [Playwright] "street photography people" → 15 images
  [Playwright] "crowd walking city" → 15 images
  [Playwright] "person portrait outdoor" → 15 images
  [Playwright] "people market" → 16 images
  [Playwright] "human activity park" → 16 images
  → 77 images collected for [person]

[Playwright] Category: car  target=100
  [Playwright] "cars on road traffic" → 15 images
  [Playwright] "car parking lot" → 15 images
  [Playwright] "automobile highway" → 16 images
  [Playwright] "vehicles city street" → 15 images
  [Playwright] "car close up" → 16 images
  → 77 images collected for [car]

[Playwright] Category: dog  target=100
  [Playwright] "dog outdoor park" → 15 images
  [Playwright] "dog running field" → 16 images
  [Playwright] "puppy playing" → 16 images
  [Playwright] "dog portrait" → 16 images
  [Playwright] "dog fetch ball" → 15 images
  → 78 images collected for [dog]

[Playwright] Total: 232 images
Pl

In [5]:
def collect_fiftyone(cfg: dict) -> list[str]:
    # Download fiftyone_per_category images per category from Open Images V7.
    all_paths = []
    per_cat   = cfg['fo_cat']

    for category in cfg['categories']:
        fo_class = cfg['fiftyone_classes'][category]
        dest_dir = Path(cfg['output_dir']) / 'openimages' / category
        dest_dir.mkdir(parents=True, exist_ok=True)

        ds_name = f"oi_{category}_{per_cat}"
        print(f'\n[FiftyOne] Category: {category}  ({fo_class})  target={per_cat}')

        try:
            # Delete stale dataset with same name if it exists
            if ds_name in fo.list_datasets():
                fo.delete_dataset(ds_name)

            dataset = foz.load_zoo_dataset(
                'open-images-v7',
                split='train',
                label_types=['detections'],
                classes=[fo_class],
                max_samples=per_cat + 20,
                only_matching=True,
                dataset_name=ds_name,
            )

            cat_paths = []
            for sample in dataset:
                src  = Path(sample.filepath)
                dest = dest_dir / src.name
                if not dest.exists():
                    shutil.copy2(src, dest)
                cat_paths.append(str(dest))
                if len(cat_paths) >= per_cat:
                    break

            fo.delete_dataset(ds_name)
            print(f'  → {len(cat_paths)} images collected for [{category}]')
            all_paths.extend(cat_paths)

        except Exception as e:
            print(f'  [FiftyOne ERROR] {category}: {e}')

    print(f'\n[FiftyOne] Total: {len(all_paths)} images')
    return all_paths


print('Running FiftyOne collection...')
fiftyone_paths  = collect_fiftyone(CFG)
all_image_paths += fiftyone_paths

print(f'  Playwright  : {len(playwright_paths)} images')
print(f'  FiftyOne    : {len(fiftyone_paths)} images')
print(f'  Total       : {len(all_image_paths)} images ')

Running FiftyOne collection...

[FiftyOne] Category: person  (Person)  target=200


INFO:fiftyone.zoo.datasets:Downloading split 'train' to '/root/fiftyone/open-images-v7/train' if necessary


INFO:fiftyone.utils.openimages:Downloading 'https://storage.googleapis.com/openimages/2018_04/train/train-images-boxable-with-rotation.csv' to '/root/fiftyone/open-images-v7/train/metadata/image_ids.csv'


 100% |██████|    4.8Gb/4.8Gb [2.7s elapsed, 0s remaining, 1.7Gb/s]        


INFO:eta.core.utils: 100% |██████|    4.8Gb/4.8Gb [2.7s elapsed, 0s remaining, 1.7Gb/s]        


...

 100% |█████████████████| 220/220 [487.3ms elapsed, 0s remaining, 451.4 samples/s]      


INFO:eta.core.utils: 100% |█████████████████| 220/220 [487.3ms elapsed, 0s remaining, 451.4 samples/s]      


Dataset 'oi_dog_200' created


INFO:fiftyone.zoo.datasets:Dataset 'oi_dog_200' created


  → 200 images collected for [dog]

[FiftyOne] Total: 600 images

── Collection Summary ──────────────────
  Playwright  : 232 images
  FiftyOne    : 600 images
  Total       : 832 images 


### Load Grounding DINO + SAM

In [12]:
import torch
from transformers import BertModel, BertConfig
from transformers.modeling_utils import ModuleUtilsMixin

# Patch 1: fix dtype issue
if not hasattr(ModuleUtilsMixin, '_is_patched'):
    old_get_ext = ModuleUtilsMixin.get_extended_attention_mask
    def patched_get_ext(self, attention_mask, input_shape, dtype=None):
        if isinstance(dtype, torch.device):
            dtype = next(self.parameters()).dtype
        return old_get_ext(self, attention_mask, input_shape, dtype)
    ModuleUtilsMixin.get_extended_attention_mask = patched_get_ext
    ModuleUtilsMixin._is_patched = True

# Patch 2: fix get_head_mask — patch the BASE class so ALL subclasses inherit it
if not hasattr(ModuleUtilsMixin, 'get_head_mask'):
    def get_head_mask(self, head_mask, num_hidden_layers, is_attention_chunked=False):
        if head_mask is not None:
            head_mask = self._convert_head_mask_to_5d(head_mask, num_hidden_layers)
            if is_attention_chunked is True:
                head_mask = head_mask.unsqueeze(-1)
        else:
            head_mask = [None] * num_hidden_layers
        return head_mask
    ModuleUtilsMixin.get_head_mask = get_head_mask

print("Compatibility patches applied.")

Compatibility patches applied.


In [14]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

def load_gdino(cfg: dict):
    model = gd_load_model(cfg['gd_config'], cfg['gd_weights'])
    return model.to(DEVICE)

def load_sam(cfg: dict) -> SamPredictor:
    sam = sam_model_registry[cfg['sam_model_type']](checkpoint=cfg['sam_checkpoint'])
    sam.to(DEVICE)
    return SamPredictor(sam)

print('Loading Grounding DINO...')
gd_model = load_gdino(CFG)
print('Loading SAM ViT-H...')
sam_predictor = load_sam(CFG)
print('Models ready.')

Device: cuda
Loading Grounding DINO...
final text_encoder_type: bert-base-uncased


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading SAM ViT-H...
Models ready.


Annotation Pipeline

For each image:
1. Resize to 640×640
2. Grounding DINO detects bounding boxes using the text prompt
3. SAM generates a segmentation mask for each box
4. Mask is encoded as RLE for COCO compatibility

In [15]:
import torch
import numpy as np
from pathlib import Path
from PIL import Image
import cv2
from transformers.modeling_utils import ModuleUtilsMixin
from transformers.models.bert.modeling_bert import BertModel

def mask_to_rle(mask: np.ndarray) -> list[int]:
    flat = mask.flatten().astype(np.uint8)
    if flat.size == 0:
        return [0]
    counts: list[int] = []
    val, run = int(flat[0]), 1
    for bit in flat[1:]:
        b = int(bit)
        if b == val:
            run += 1
        else:
            counts.append(run)
            run, val = 1, b
    counts.append(run)
    return counts

def annotate_image(
    image_path: str,
    gd_model,
    sam_predictor,
    cfg: dict,
) -> tuple[dict | None, list[dict]]:

    bgr = cv2.imread(image_path)
    if bgr is None:
        print(f'   Cannot read: {image_path}')
        return None, []

    W, H = cfg['image_size']
    bgr_resized  = cv2.resize(bgr, (W, H))
    rgb_resized  = cv2.cvtColor(bgr_resized, cv2.COLOR_BGR2RGB)
    pil_image    = Image.fromarray(rgb_resized)

    # Save temp file for Grounding DINO loader
    tmp_path = '/tmp/_gd_tmp.jpg'
    pil_image.save(tmp_path)
    _, image_tensor = gd_load_image(tmp_path)

    # Predict with GroundingDINO
    boxes, logits, phrases = gd_predict(
        model=gd_model,
        image=image_tensor,
        caption=cfg['gd_text_prompt'],
        box_threshold=cfg['gd_box_threshold'],
        text_threshold=cfg['gd_text_threshold'],
        device=DEVICE,
    )

    image_info = {
        'file_name': Path(image_path).name,
        'height':    H,
        'width':     W,
    }

    if boxes is None or len(boxes) == 0:
        return image_info, []

    # Move boxes to CPU before conversion to numpy
    boxes_cpu = boxes.cpu().numpy()
    # Move logits to CPU before conversion to numpy
    logits_cpu = logits.cpu().numpy()

    # SAM segmentation
    sam_predictor.set_image(rgb_resized)
    annotations: list[dict] = []

    # Use range to ensure we sync boxes, phrases, and logits correctly
    for i in range(len(boxes_cpu)):
        box = boxes_cpu[i]
        phrase = phrases[i]
        logit = logits_cpu[i]

        cx, cy, bw, bh = box
        x1 = max(0, int((cx - bw / 2) * W))
        y1 = max(0, int((cy - bh / 2) * H))
        x2 = min(W, int((cx + bw / 2) * W))
        y2 = min(H, int((cy + bh / 2) * H))

        if x2 - x1 >= 2 and y2 - y1 >= 2:
            masks, scores, _ = sam_predictor.predict(
                box=np.array([x1, y1, x2, y2]),
                multimask_output=True,
            )
            best_mask = masks[int(np.argmax(scores))]

            annotations.append({
                '_phrase': phrase,
                '_score':  float(logit),
                'bbox':    [x1, y1, x2 - x1, y2 - y1],
                'area':    int((x2 - x1) * (y2 - y1)),
                'segmentation': {
                    'size':   [H, W],
                    'counts': mask_to_rle(best_mask),
                },
                'iscrowd': 0,
            })

    return image_info, annotations


In [16]:
print('Zero-Shot Annotation')

image_infos:     list[dict]       = []
all_annotations: list[list[dict]] = []

for i, img_path in enumerate(all_image_paths):
    print(f'  [{i+1}/{len(all_image_paths)}] {Path(img_path).name}', end='  ')
    img_info, anns = annotate_image(img_path, gd_model, sam_predictor, CFG)
    if img_info is None:
        continue
    image_infos.append(img_info)
    all_annotations.append(anns)
    print(f'→ {len(anns)} annotations')

print(f'\nAnnotation complete: {len(image_infos)} images processed.')

Zero-Shot Annotation
  [1/832] street_photography_people_0000.jpg  

→ 3 annotations
  [2/832] street_photography_people_0001.jpg  → 7 annotations
  [3/832] street_photography_people_0002.jpg  → 2 annotations
  [4/832] street_photography_people_0003.jpg  → 4 annotations
  [5/832] street_photography_people_0004.jpg  → 8 annotations
  [6/832] street_photography_people_0005.jpg  → 2 annotations
  [7/832] street_photography_people_0006.jpg  → 2 annotations
  [8/832] street_photography_people_0007.jpg  → 2 annotations
  [9/832] street_photography_people_0008.jpg  → 2 annotations
  [10/832] street_photography_people_0009.jpg  → 1 annotations
  [11/832] street_photography_people_0010.jpg  → 1 annotations
  [12/832] street_photography_people_0011.jpg  → 1 annotations
  [13/832] street_photography_people_0012.jpg  → 2 annotations
  [14/832] street_photography_people_0013.jpg  → 3 annotations
  [15/832] street_photography_people_0014.jpg  → 2 annotations
  [16/832] crowd_walking_city_0000.jpg  → 6 annotations
  [17/832] crowd_walking_city_0001.jpg  → 3 annotation

# Assemble & Validate COCO JSON


In [17]:
def build_coco(
    image_infos:     list[dict],
    all_annotations: list[list[dict]],
) -> dict:

    # Derive category list from detected phrases
    phrase_set: set[str] = set()
    for anns in all_annotations:
        for a in anns:
            phrase_set.add(a['_phrase'])

    categories = [
        {'id': i + 1, 'name': phrase, 'supercategory': 'object'}
        for i, phrase in enumerate(sorted(phrase_set))
    ]
    phrase_to_id = {c['name']: c['id'] for c in categories}

    coco_images:      list[dict] = []
    coco_annotations: list[dict] = []
    ann_id = 1

    for img_id, (img_info, anns) in enumerate(zip(image_infos, all_annotations), start=1):
        coco_images.append({
            'id':        img_id,
            'file_name': img_info['file_name'],
            'height':    img_info['height'],
            'width':     img_info['width'],
        })
        for ann in anns:
            coco_annotations.append({
                'id':           ann_id,
                'image_id':     img_id,
                'category_id':  phrase_to_id[ann['_phrase']],
                'bbox':         ann['bbox'],
                'area':         ann['area'],
                'segmentation': ann['segmentation'],
                'iscrowd':      ann['iscrowd'],
                'score':        ann['_score'],
            })
            ann_id += 1

    return {
        'info': {
            'description': 'Augle AI Assignment',
            'version':     '1.0',
            'year':        2025,
        },
        'licenses':    [],
        'images':      coco_images,
        'annotations': coco_annotations,
        'categories':  categories,
    }


def validate_coco(coco: dict) -> bool:
    required_top = {'info', 'images', 'annotations', 'categories'}
    missing = required_top - set(coco.keys())
    if missing:
        raise ValueError(f'Missing top-level keys: {missing}')

    img_ids = {img['id'] for img in coco['images']}
    cat_ids = {cat['id'] for cat in coco['categories']}

    for ann in coco['annotations']:
        required_ann = ('id', 'image_id', 'category_id', 'bbox', 'area',
                        'segmentation', 'iscrowd')
        for key in required_ann:
            if key not in ann:
                raise ValueError(f'Annotation {ann.get("id")} missing key "{key}"')
        if ann['image_id'] not in img_ids:
            raise ValueError(f'Annotation {ann["id"]} → unknown image_id {ann["image_id"]}')
        if ann['category_id'] not in cat_ids:
            raise ValueError(f'Annotation {ann["id"]} → unknown category_id {ann["category_id"]}')
        bbox = ann['bbox']
        if len(bbox) != 4 or bbox[2] <= 0 or bbox[3] <= 0:
            raise ValueError(f'Annotation {ann["id"]} has invalid bbox {bbox}')

    print(
        f'[COCO Validation] — '
        f'{len(coco["images"])} images, '
        f'{len(coco["annotations"])} annotations, '
        f'{len(coco["categories"])} categories.'
    )
    return True


# Build, validate, and save
coco = build_coco(image_infos, all_annotations)
validate_coco(coco)

out_path = Path(CFG['coco_path'])
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, 'w') as f:
    json.dump(coco, f, indent=2)

print(f'\nCOCO JSON saved → {out_path}')
print(f'  Total images annotated : {len(coco["images"])}')
print(f'  Total annotations      : {len(coco["annotations"])}')
print(f'  Categories             : {[c["name"] for c in coco["categories"]]}')

[COCO Validation] PASSED — 832 images, 2897 annotations, 4 categories.

COCO JSON saved → /content/drive/MyDrive/augle_ai/data/annotations/instances.json
  Total images annotated : 832
  Total annotations      : 2897
  Categories             : ['car', 'dog', 'person', 'person dog']


# Class Distribution

In [18]:
from collections import Counter

id_to_name = {c['id']: c['name'] for c in coco['categories']}
counts     = Counter(ann['category_id'] for ann in coco['annotations'])

print('Class distribution:')
for cat_id, count in sorted(counts.items()):
    print(f'  {id_to_name[cat_id]:<20} : {count} annotations')

Class distribution:
  car                  : 705 annotations
  dog                  : 416 annotations
  person               : 1769 annotations
  person dog           : 7 annotations


#  Augmentation Helpers

In [19]:
import random


def resize_with_annotations(
    image:       np.ndarray,
    bboxes:      list[list[float]],
    masks:       list[np.ndarray],
    target_size: tuple[int, int],
) -> tuple[np.ndarray, list[list[float]], list[np.ndarray]]:
    h, w = image.shape[:2]
    W, H = target_size
    image  = cv2.resize(image, (W, H), interpolation=cv2.INTER_LINEAR)
    sx, sy = W / w, H / h
    new_bboxes = [[x * sx, y * sy, bw * sx, bh * sy] for x, y, bw, bh in bboxes]
    new_masks  = [
        cv2.resize(m.astype(np.uint8), (W, H),
                   interpolation=cv2.INTER_NEAREST).astype(bool)
        for m in masks
    ]
    return image, new_bboxes, new_masks


def normalize(image: np.ndarray) -> np.ndarray:
    image = image.astype(np.float32) / 255.0
    mean  = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std   = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    return (image - mean) / std


def random_horizontal_flip(
    image:  np.ndarray,
    bboxes: list[list[float]],
    masks:  list[np.ndarray],
    p: float = 0.5,
) -> tuple[np.ndarray, list[list[float]], list[np.ndarray]]:
    if random.random() > p:
        return image, bboxes, masks
    h, w   = image.shape[:2]
    image  = cv2.flip(image, 1)
    new_bboxes = [[w - x - bw, y, bw, bh] for x, y, bw, bh in bboxes]
    new_masks  = [cv2.flip(m.astype(np.uint8), 1).astype(bool) for m in masks]
    return image, new_bboxes, new_masks


def random_affine(
    image:         np.ndarray,
    bboxes:        list[list[float]],
    masks:         list[np.ndarray],
    max_angle:     float = 15.0,
    max_translate: float = 0.1,
    scale_range:   tuple[float, float] = (0.9, 1.1),
) -> tuple[np.ndarray, list[list[float]], list[np.ndarray]]:
    """Random rotation + translation + scale. Bboxes re-derived from corner transforms."""
    h, w   = image.shape[:2]
    angle  = random.uniform(-max_angle, max_angle)
    tx     = random.uniform(-max_translate, max_translate) * w
    ty     = random.uniform(-max_translate, max_translate) * h
    scale  = random.uniform(*scale_range)
    cx, cy = w / 2.0, h / 2.0

    M = cv2.getRotationMatrix2D((cx, cy), angle, scale)
    M[0, 2] += tx
    M[1, 2] += ty

    image = cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_LINEAR,
                           borderMode=cv2.BORDER_REFLECT_101)

    def transform_point(px, py):
        return (
            M[0, 0] * px + M[0, 1] * py + M[0, 2],
            M[1, 0] * px + M[1, 1] * py + M[1, 2],
        )

    new_bboxes: list[list[float]] = []
    for x, y, bw, bh in bboxes:
        corners = [
            transform_point(x,      y),
            transform_point(x + bw, y),
            transform_point(x,      y + bh),
            transform_point(x + bw, y + bh),
        ]
        xs = [c[0] for c in corners]; ys = [c[1] for c in corners]
        nx  = max(0.0,     min(xs));  ny  = max(0.0,     min(ys))
        nx2 = min(float(w), max(xs)); ny2 = min(float(h), max(ys))
        new_bboxes.append([nx, ny, nx2 - nx, ny2 - ny])

    new_masks = [
        cv2.warpAffine(m.astype(np.uint8), M, (w, h),
                       flags=cv2.INTER_NEAREST,
                       borderMode=cv2.BORDER_CONSTANT, borderValue=0).astype(bool)
        for m in masks
    ]
    return image, new_bboxes, new_masks


def color_jitter(image: np.ndarray) -> np.ndarray:
    """Random brightness, contrast, and HSV saturation jitter."""
    alpha = random.uniform(0.7, 1.3)
    beta  = random.uniform(-30, 30)
    image = cv2.convertScaleAbs(image, alpha=alpha, beta=beta)
    hsv   = cv2.cvtColor(image, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[:, :, 1] = np.clip(hsv[:, :, 1] * random.uniform(0.7, 1.3), 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)


def random_crop(
    image:      np.ndarray,
    bboxes:     list[list[float]],
    masks:      list[np.ndarray],
    crop_ratio: float = 0.9,
) -> tuple[np.ndarray, list[list[float]], list[np.ndarray]]:
    """Crop retaining ≥ crop_ratio of each dimension. Drops boxes clipped below 5px."""
    h, w  = image.shape[:2]
    new_w = int(w * random.uniform(crop_ratio, 1.0))
    new_h = int(h * random.uniform(crop_ratio, 1.0))
    x0    = random.randint(0, w - new_w)
    y0    = random.randint(0, h - new_h)
    image = image[y0:y0 + new_h, x0:x0 + new_w]

    new_bboxes: list[list[float]] = []
    new_masks:  list[np.ndarray]  = []
    for (x, y, bw, bh), mask in zip(bboxes, masks):
        nx  = max(0.0,          x  - x0)
        ny  = max(0.0,          y  - y0)
        nx2 = min(float(new_w), x + bw - x0)
        ny2 = min(float(new_h), y + bh - y0)
        if nx2 - nx > 5 and ny2 - ny > 5:
            new_bboxes.append([nx, ny, nx2 - nx, ny2 - ny])
            new_masks.append(mask[y0:y0 + new_h, x0:x0 + new_w])

    return image, new_bboxes, new_masks


print('Augmentation helpers defined.')

Augmentation helpers defined.


# RLE Decoder + COCOObjectDataset

In [20]:
from torch.utils.data import Dataset, DataLoader


def rle_to_mask(rle_counts: list[int], height: int, width: int) -> np.ndarray:
    """Decode a flat RLE list back to a boolean 2D mask."""
    flat: list[int] = []
    val = 0
    for count in rle_counts:
        flat.extend([val] * count)
        val = 1 - val
    arr   = np.array(flat, dtype=np.uint8)
    total = height * width
    if arr.size < total:
        arr = np.pad(arr, (0, total - arr.size))
    return arr[:total].reshape(height, width).astype(bool)


class COCOObjectDataset(Dataset):
    """
    Loads COCO-format annotations and returns
        image   : FloatTensor [3, H, W]  (ImageNet-normalised)
        label   : LongTensor  []          (0-indexed primary category)
        bboxes  : FloatTensor [N, 4]      (x, y, w, h — pixel coords)
        masks   : BoolTensor  [N, H, W]
    """

    def __init__(
        self,
        coco_path:   str,
        image_root:  str,
        target_size: tuple[int, int] = (224, 224),
        augment:     bool = True,
        max_masks:   int  = 10,
    ):
        self.image_root  = Path(image_root)
        self.target_size = target_size    # (W, H)
        self.augment     = augment
        self.max_masks   = max_masks

        with open(coco_path) as f:
            coco = json.load(f)

        self.categories  = {c['id']: c['name'] for c in coco['categories']}
        self.num_classes = len(self.categories)
        sorted_ids       = sorted(self.categories.keys())
        self.id_to_idx   = {cid: i for i, cid in enumerate(sorted_ids)}

        ann_by_image: dict[int, list[dict]] = {}
        for ann in coco['annotations']:
            ann_by_image.setdefault(ann['image_id'], []).append(ann)

        self.samples: list[dict] = []
        for img in coco['images']:
            anns = ann_by_image.get(img['id'], [])
            if not anns:
                continue
            img_path = self.image_root / img['file_name']
            if not img_path.exists():
                candidates = list(self.image_root.rglob(img['file_name']))
                if not candidates:
                    continue
                img_path = candidates[0]
            self.samples.append({
                'path':   str(img_path),
                'height': img['height'],
                'width':  img['width'],
                'anns':   anns,
            })

        print(
            f'[Dataset] {len(self.samples)} images | '
            f'{self.num_classes} categories | augment={augment}'
        )

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> dict:
        sample = self.samples[idx]
        bgr    = cv2.imread(sample['path'])
        W_out, H_out = self.target_size

        # Graceful fallback for missing files
        if bgr is None:
            return {
                'image':  torch.zeros(3, H_out, W_out),
                'label':  torch.tensor(0, dtype=torch.long),
                'bboxes': torch.zeros(1, 4),
                'masks':  torch.zeros(1, H_out, W_out, dtype=torch.bool),
            }

        image  = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        anns   = sample['anns'][:self.max_masks]
        bboxes = [a['bbox'] for a in anns]

        masks: list[np.ndarray] = []
        for a in anns:
            seg = a['segmentation']
            if isinstance(seg, dict) and 'counts' in seg:
                mask = rle_to_mask(seg['counts'], seg['size'][0], seg['size'][1])
            else:
                mask = np.zeros((sample['height'], sample['width']), dtype=bool)
            masks.append(mask)

        # Resize first
        image, bboxes, masks = resize_with_annotations(
            image, bboxes, masks, self.target_size
        )

        if self.augment:
            image = color_jitter(image)
            image, bboxes, masks = random_horizontal_flip(image, bboxes, masks)
            image, bboxes, masks = random_affine(image, bboxes, masks)
            image, bboxes, masks = random_crop(image, bboxes, masks)
            # Resize back to target after crop
            image, bboxes, masks = resize_with_annotations(
                image, bboxes, masks, self.target_size
            )

        image_norm   = normalize(image)
        image_tensor = torch.from_numpy(image_norm).permute(2, 0, 1).float()

        if bboxes:
            bbox_tensor = torch.tensor(bboxes, dtype=torch.float32)
            mask_tensor = torch.tensor(
                np.stack([m.astype(np.uint8) for m in masks], axis=0),
                dtype=torch.bool,
            )
        else:
            bbox_tensor = torch.zeros(1, 4,              dtype=torch.float32)
            mask_tensor = torch.zeros(1, H_out, W_out,   dtype=torch.bool)

        primary_cat_id = anns[0]['category_id'] if anns else sorted(self.id_to_idx)[0]
        label = torch.tensor(self.id_to_idx[primary_cat_id], dtype=torch.long)

        return {
            'image':  image_tensor,
            'label':  label,
            'bboxes': bbox_tensor,
            'masks':  mask_tensor,
        }


print('COCOObjectDataset defined.')

COCOObjectDataset defined.


#Collate Function & DataLoader Factory

In [21]:
def collate_fn(batch: list[dict]) -> dict:
    """Collate variable-N-annotation samples into a batch."""
    return {
        'image':  torch.stack([b['image']  for b in batch]),
        'label':  torch.stack([b['label']  for b in batch]),
        'bboxes': [b['bboxes'] for b in batch],
        'masks':  [b['masks']  for b in batch],
    }


def get_dataloaders(
    coco_path:   str,
    image_root:  str,
    batch_size:  int   = 16,
    val_split:   float = 0.15,
    target_size: tuple[int, int] = (224, 224),
    num_workers: int   = 2,
) -> tuple[DataLoader, DataLoader, int]:
    """
    Returns (train_loader, val_loader, num_classes).

    Fix: separate dataset instances so augmentation does NOT leak into validation.
    """
    # Use a plain dataset to get stable index list
    base_ds = COCOObjectDataset(
        coco_path, image_root, target_size=target_size, augment=False
    )
    n_total = len(base_ds)
    n_val   = max(1, int(n_total * val_split))
    n_train = n_total - n_val

    gen = torch.Generator().manual_seed(42)
    train_indices, val_indices = torch.utils.data.random_split(
        range(n_total), [n_train, n_val], generator=gen
    )

    train_ds_full = COCOObjectDataset(
        coco_path, image_root, target_size=target_size, augment=True
    )
    val_ds_full = COCOObjectDataset(
        coco_path, image_root, target_size=target_size, augment=False
    )

    train_ds = torch.utils.data.Subset(train_ds_full, list(train_indices))
    val_ds   = torch.utils.data.Subset(val_ds_full,   list(val_indices))

    _pin = torch.cuda.is_available()
    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=_pin, collate_fn=collate_fn,
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=_pin, collate_fn=collate_fn,
    )
    return train_loader, val_loader, base_ds.num_classes


print('DataLoader factory defined.')

DataLoader factory defined.


# Verify Dataset + DataLoader

In [22]:
COCO_PATH  = CFG['coco_path']
IMAGE_ROOT = CFG['output_dir']

train_loader, val_loader, num_classes = get_dataloaders(
    coco_path=COCO_PATH,
    image_root=IMAGE_ROOT,
    batch_size=16,
    val_split=0.15,
    target_size=(224, 224),
    num_workers=2,
)

print(f'\nnum_classes : {num_classes}')
print(f'train batches: {len(train_loader)}')
print(f'val   batches: {len(val_loader)}')

# Inspect one batch
batch = next(iter(train_loader))
print(f'\nSample batch:')
print(f'  image  : {batch["image"].shape}  dtype={batch["image"].dtype}')
print(f'  label  : {batch["label"].shape}   values={batch["label"].tolist()}')
print(f'  bboxes : list of {len(batch["bboxes"])} tensors, first={batch["bboxes"][0].shape}')
print(f'  masks  : list of {len(batch["masks"])}  tensors, first={batch["masks"][0].shape}')

[Dataset] 809 images | 4 categories | augment=False
[Dataset] 809 images | 4 categories | augment=True
[Dataset] 809 images | 4 categories | augment=False

num_classes : 4
train batches: 43
val   batches: 8

Sample batch:
  image  : torch.Size([16, 3, 224, 224])  dtype=torch.float32
  label  : torch.Size([16])   values=[2, 1, 1, 2, 0, 1, 2, 2, 2, 2, 2, 2, 1, 2, 1, 2]
  bboxes : list of 16 tensors, first=torch.Size([1, 4])
  masks  : list of 16  tensors, first=torch.Size([1, 224, 224])
